# Partition Cell Types Deepclean

Perform deepcleaning removal of doublet clusters based on manual annotation

Relabel L3 AIFI cell type labels based on deepcleaning manual annotations

## Load libraries

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
from datetime import date
import hisepy
import os
import pandas as pd
import re
import scanpy as sc
import scanpy.external as sce
import glob


In [108]:
out_dir = 'deepcleaned_output'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

## Helper functions

These make it a bit simpler to cache and read in files from HISE

In [3]:
def cache_uuid_path(uuid):
    cache_path = '/home/jupyter/cache/{u}'.format(u = uuid)
    if not os.path.isdir(cache_path):
        hise_res = hisepy.reader.cache_files([uuid])
    filename = os.listdir(cache_path)[0]
    cache_file = '{p}/{f}'.format(p = cache_path, f = filename)
    return cache_file

In [4]:
def read_parquet_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = pd.read_parquet(cache_file)
    return res

In [5]:
def sort_adata_uuid(uuid, sort_cols = ['AIFI_L2', 'sample.sampleKitGuid']):
    cache_file = cache_uuid_path(uuid)
    adata = sc.read_h5ad(cache_file)
    obs = adata.obs
    obs = obs.sort_values(sort_cols)
    adata = adata[obs.index]
    adata.write_h5ad(cache_file)

This function will enable us to connect to our .h5ad files without loading the entire thing into memory. We'll then load only the cells that we want for each cell class to assemble them for writing. This should save us some overhead as we do our subsetting.

In [6]:
def read_adata_backed_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = sc.read_h5ad(cache_file, backed = 'r')
    return res

This function will remove Ig-related genes, which is recommended by Marla Glass for analysis of B cell subtypes

In [7]:
def remove_ig_genes(adata):
    igl_genes = [gene for gene in adata.var_names if gene.startswith("IGL")]
    igk_genes = [gene for gene in adata.var_names if gene.startswith("IGK")]
    ighc_genes = [gene for gene in adata.var_names if gene.startswith("IGH")]
    exl_genes = igl_genes + igk_genes + ighc_genes

    filtered_genes = [gene for gene in adata.var_names if gene not in exl_genes]
    adata = adata[:, filtered_genes]

    return adata

This function will apply a standard normalization, nearest neighbors, clustering, and UMAP process to our cell subsets:

In [8]:
def process_adata(adata, resolution = 2):
    
    # Keep a copy of the raw data
    adata = adata.raw.to_adata()
    adata.raw = adata

    print('Normalizing', end = "; ")
    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)

    print('Finding HVGs', end = "; ")
    # Restrict downstream steps to variable genes
    sc.pp.highly_variable_genes(adata)
    adata = adata[:, adata.var_names[adata.var['highly_variable']]].copy()

    print('Scaling', end = "; ")
    # Scale variable genes
    sc.pp.scale(adata)

    print('PCA', end = "; ")
    # Run PCA
    sc.tl.pca(adata, svd_solver = 'arpack')
    
    print('Neighbors', end = "; ")
    # Find nearest neighbors
    sc.pp.neighbors(
        adata, 
        n_neighbors = 50,
        n_pcs = 30
    )

    print('Leiden', end = "; ")
    # Find clusters
    sc.tl.leiden(
        adata, 
        resolution = resolution, 
        key_added = 'leiden_{r}'.format(r = resolution),
        n_iterations = 2
    )

    print('UMAP', end = "; ")
    # Run UMAP
    sc.tl.umap(adata, min_dist = 0.05)
    
    print('Renormalizing')
    adata = adata.raw.to_adata()
    adata.raw = adata

    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    
    return adata

In [9]:
def format_cell_type(cell_type):
    cell_type = re.sub('\\+', 'pos', cell_type)
    cell_type = re.sub('-', 'neg', cell_type)
    cell_type = re.sub(' ', '_', cell_type)
    return cell_type

In [10]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

In [11]:
# make a function to find files
def get_filepaths_with_glob(root_path: str, file_regex: str):
    return glob.glob(os.path.join(root_path, file_regex))

In [31]:
# Define a function to extract the desired substring using regex
def extract_substring(path, pattern = r'up1_cluster_unharmonize_(.*?)_\d{4}-\d{2}-\d{2}\.h5ad'):
    #pattern = r'preRA_cluster_harmonize_(.*?)_\d{4}-\d{2}-\d{2}\.h5ad'
    match = re.search(pattern, path)
    if match:
        return match.group(1)
    return None

In [14]:
def read_anndata_files(file_tuples):
    """
    Read Anndata objects from H5AD files and store them in a dictionary with custom names.

    Parameters:
        file_tuples (list of tuples): List of tuples where each tuple contains filename and desired name.

    Returns:
        dict: Dictionary containing Anndata objects with custom names.
    """
    anndata_dict = {}
    for filename, name in file_tuples:
        anndata_obj = anndata.read_h5ad(filename)
        anndata_dict[name] = anndata_obj
    return anndata_dict

In [15]:
def reformat_cell_type(cell_type):
    '''convert cell type names read in from file back to original L3 labels'''
    cell_type = re.sub('pos', '+', cell_type)
    cell_type = re.sub('neg', '-', cell_type)
    cell_type = re.sub('_',' ', cell_type)
    return cell_type

## Identify files for use in HISE

In [16]:
search_id = 'sodium-gadolinium-silicon'

Retrieve files stored in our HISE project store

In [17]:
ps_df = hisepy.list_files_in_project_store('Dyna_IHandA')
ps_df = ps_df[['id', 'name']]

Filter for files from the previous notebook using our search_id

In [18]:
search_df = ps_df[ps_df['name'].str.contains(search_id)]
search_df = search_df.sort_values('name')

In [19]:
h5ad_df = search_df[search_df['name'].str.contains('.h5ad')]

In [20]:
h5ad_df

,id,name
218,7467d353-d8bf-467e-911c-4f759b83caf7,sodium-gadolinium-silicon/up1_cluster_unharmon...
219,eebab05a-bd3c-4921-8ae3-7345fac966c9,sodium-gadolinium-silicon/up1_cluster_unharmon...
220,410fc93a-f497-4ea4-81e2-fb8cbdc79a28,sodium-gadolinium-silicon/up1_cluster_unharmon...
221,6b18c554-64b3-43ae-9514-6ecd18c30f4d,sodium-gadolinium-silicon/up1_cluster_unharmon...
222,6f5b21ba-d98b-4003-bcf0-30e26a2cc2df,sodium-gadolinium-silicon/up1_cluster_unharmon...
...,...,...
213,e71668a5-f418-47d1-add1-40540e6b81d3,sodium-gadolinium-silicon/up1_cluster_unharmon...
214,4aa69351-6c4e-4527-83bd-b34e0e5127e6,sodium-gadolinium-silicon/up1_cluster_unharmon...
215,4a18d4bc-46a6-4946-8997-ef65d3e7144f,sodium-gadolinium-silicon/up1_cluster_unharmon...
216,04e4ecdd-4aeb-4f29-9326-94078d132295,sodium-gadolinium-silicon/up1_cluster_unharmon...


In [81]:
h5ad_uuids = {}
for i in range(h5ad_df.shape[0]):
    fn = h5ad_df['name'].tolist()[i]
    group_name = re.sub('.+pbmc_', '', fn)
    group_name = re.sub('_init.+', '', group_name)
    h5ad_uuids[group_name] = h5ad_df['id'].tolist()[i]

In [82]:
h5ad_uuids

{'sodium-gadolinium-silicon/up1_cluster_unharmonize_ASDC_2024-09-30.h5ad': '7467d353-d8bf-467e-911c-4f759b83caf7',
 'sodium-gadolinium-silicon/up1_cluster_unharmonize_Activated_memory_B_cell_2024-09-30.h5ad': 'eebab05a-bd3c-4921-8ae3-7345fac966c9',
 'sodium-gadolinium-silicon/up1_cluster_unharmonize_Adaptive_NK_cell_2024-09-30.h5ad': '410fc93a-f497-4ea4-81e2-fb8cbdc79a28',
 'sodium-gadolinium-silicon/up1_cluster_unharmonize_BaEoMaP_cell_2024-09-30.h5ad': '6b18c554-64b3-43ae-9514-6ecd18c30f4d',
 'sodium-gadolinium-silicon/up1_cluster_unharmonize_C1Qpos_CD16_monocyte_2024-09-30.h5ad': '6f5b21ba-d98b-4003-bcf0-30e26a2cc2df',
 'sodium-gadolinium-silicon/up1_cluster_unharmonize_CD14pos_cDC2_2024-09-30.h5ad': '153e45fc-e772-497c-9c4b-e986b7885f01',
 'sodium-gadolinium-silicon/up1_cluster_unharmonize_CD27neg_effector_B_cell_2024-09-30.h5ad': '9acd576f-2271-4ddb-b93e-20f0f63f87af',
 'sodium-gadolinium-silicon/up1_cluster_unharmonize_CD27pos_effector_B_cell_2024-09-30.h5ad': 'fcafe12f-3a15-4807

In [83]:
len(h5ad_uuids)

70

In [76]:
#removing up1_cluster_unharmonize_KLRB1pos_memory_CD8_Treg_2024-09-30.h5ad from the list
h5ad_uuids.pop('sodium-gadolinium-silicon/up1_cluster_unharmonize_KLRB1pos_memory_CD8_Treg_2024-09-30.h5ad')
h5ad_uuids

{'sodium-gadolinium-silicon/up1_cluster_unharmonize_ASDC_2024-09-30.h5ad': '7467d353-d8bf-467e-911c-4f759b83caf7',
 'sodium-gadolinium-silicon/up1_cluster_unharmonize_Activated_memory_B_cell_2024-09-30.h5ad': 'eebab05a-bd3c-4921-8ae3-7345fac966c9',
 'sodium-gadolinium-silicon/up1_cluster_unharmonize_Adaptive_NK_cell_2024-09-30.h5ad': '410fc93a-f497-4ea4-81e2-fb8cbdc79a28',
 'sodium-gadolinium-silicon/up1_cluster_unharmonize_BaEoMaP_cell_2024-09-30.h5ad': '6b18c554-64b3-43ae-9514-6ecd18c30f4d',
 'sodium-gadolinium-silicon/up1_cluster_unharmonize_C1Qpos_CD16_monocyte_2024-09-30.h5ad': '6f5b21ba-d98b-4003-bcf0-30e26a2cc2df',
 'sodium-gadolinium-silicon/up1_cluster_unharmonize_CD14pos_cDC2_2024-09-30.h5ad': '153e45fc-e772-497c-9c4b-e986b7885f01',
 'sodium-gadolinium-silicon/up1_cluster_unharmonize_CD27neg_effector_B_cell_2024-09-30.h5ad': '9acd576f-2271-4ddb-b93e-20f0f63f87af',
 'sodium-gadolinium-silicon/up1_cluster_unharmonize_CD27pos_effector_B_cell_2024-09-30.h5ad': 'fcafe12f-3a15-4807

## Download and sort .h5ad files

Sorting the .h5ad files by AIFI_L2 will make reading each cell type much faster by placing cells of the same type next to each other in the sparse matrix.

In [80]:
# run once
for uuid in h5ad_uuids.values():
    sort_adata_uuid(uuid, sort_cols = ['AIFI_L3'])

downloading fileID: 7467d353-d8bf-467e-911c-4f759b83caf7
Files have been successfully downloaded!
downloading fileID: eebab05a-bd3c-4921-8ae3-7345fac966c9
Files have been successfully downloaded!
downloading fileID: 410fc93a-f497-4ea4-81e2-fb8cbdc79a28
Files have been successfully downloaded!
downloading fileID: 6b18c554-64b3-43ae-9514-6ecd18c30f4d
Files have been successfully downloaded!
downloading fileID: 6f5b21ba-d98b-4003-bcf0-30e26a2cc2df
Files have been successfully downloaded!
downloading fileID: 153e45fc-e772-497c-9c4b-e986b7885f01
Files have been successfully downloaded!
downloading fileID: 9acd576f-2271-4ddb-b93e-20f0f63f87af
Files have been successfully downloaded!
downloading fileID: fcafe12f-3a15-4807-aaea-e93f08445d02
Files have been successfully downloaded!
downloading fileID: 66122892-cddf-4c40-88b2-eaa53b7a6f4c
Files have been successfully downloaded!
downloading fileID: c9356501-0f48-4590-83ea-34da6dc72b56
Files have been successfully downloaded!
downloading fileID: 

## Open connections to .h5ad files

Now that they're sorted, we can open these files with on-disk backing so we don't have to read the entire file at once.

## Process files

In [84]:
input_path = "/home/jupyter/certpro_up1/unharmonized_output/"

In [85]:
### read in processed leiden adata
filenames = get_filepaths_with_glob(input_path, "up1_cluster_unharmonize_*.h5ad")  
filenames[:5]
len(filenames)

70

In [86]:
### extract cell types
# Apply the function to each filename in the list using list comprehension
cell_types = [extract_substring(path) for path in filenames]
cell_types[:5]

['Core_CD14_monocyte',
 'KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell',
 'CD14pos_cDC2',
 'GZMKpos_CD27pos_EM_CD8_T_cell',
 'ISGpos_MAIT']

In [87]:
file_dict = dict(zip(cell_types, filenames))
list(file_dict.items())[:10]

[('Core_CD14_monocyte',
  '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_Core_CD14_monocyte_2024-09-30.h5ad'),
 ('KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell',
  '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell_2024-09-30.h5ad'),
 ('CD14pos_cDC2',
  '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_CD14pos_cDC2_2024-09-30.h5ad'),
 ('GZMKpos_CD27pos_EM_CD8_T_cell',
  '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_GZMKpos_CD27pos_EM_CD8_T_cell_2024-09-30.h5ad'),
 ('ISGpos_MAIT',
  '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_ISGpos_MAIT_2024-09-30.h5ad'),
 ('CD56bright_NK_cell',
  '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_CD56bright_NK_cell_2024-09-30.h5ad'),
 ('CM_CD8_T_cell',
  '/home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_CM_CD8_T_cell_2024-09-30.h5ad'),
 ('SOX4pos_naive_CD4_T_cell',

In [88]:
#### read in doublet csv
doublet_df = pd.read_csv("/home/jupyter/certpro_up1/final_UP1_doublet - Sheet1-2.csv")
### select only cell type and doublet leiden
doublet_df = doublet_df[["AIFI_L3", "leiden (cluster to be removed)"]]
# reformat cell types
doublet_df['AIFI_L3'] = [format_cell_type(cell_type) for cell_type in doublet_df['AIFI_L3']]
doublet_df

,AIFI_L3,leiden (cluster to be removed)
0,ASDC,"12,15"
1,Activated_memory_B_cell,"0,8"
2,Adaptive_NK_cell,"14,19,20,21,17"
3,C1Qpos_CD16_monocyte,"1,11,16,13,18"
4,CD14pos_cDC2,"1,15,20,22,23,21"
...,...,...
61,SOX4pos_naive_CD8_T_cell,"15,18,13"
62,Transitional_B_cell,"20,23,22,21,24,14"
63,Type_2_polarized_memory_B_cell,"17,8"
64,cDC1,"15,10,9,16"


In [89]:
doublet_df['AIFI_L3']

0                               ASDC
1            Activated_memory_B_cell
2                   Adaptive_NK_cell
3               C1Qpos_CD16_monocyte
4                       CD14pos_cDC2
                   ...              
61          SOX4pos_naive_CD8_T_cell
62               Transitional_B_cell
63    Type_2_polarized_memory_B_cell
64                              cDC1
65                               pDC
Name: AIFI_L3, Length: 66, dtype: object

## Store doublet clusters

In [90]:
# Convert the DataFrame to a dictionary
doublet_cluster_dict = {}

for index, row in doublet_df.iterrows():
    key = row['AIFI_L3']
    # Check if the value is not NaN or None
    if pd.notna(row['leiden (cluster to be removed)']):
        values = row['leiden (cluster to be removed)'].split(',')
    else:
        values = []
    doublet_cluster_dict[key] = values

#print(doublet_cluster_dict)

In [91]:
doublet_cluster_dict

{'ASDC': ['12', '15'],
 'Activated_memory_B_cell': ['0', '8'],
 'Adaptive_NK_cell': ['14', '19', '20', '21', '17'],
 'C1Qpos_CD16_monocyte': ['1', '11', '16', '13', '18'],
 'CD14pos_cDC2': ['1', '15', '20', '22', '23', '21'],
 'CD27pos_effector_B_cell': ['17', '21', '20'],
 'CD27neg_effector_B_cell': ['15', '17', '22', '14', '19'],
 'CD4_MAIT': ['13', '16', '4', '2', '9', '7', '16', '3', '6', '15', '17', '5'],
 'CD56bright_NK_cell': ['7', '21', '22', '24', '26'],
 'CD8_MAIT': ['19', '21', '12'],
 'CD95_memory_B_cell': ['8', '13', '15'],
 'CLP_cell': ['8', '0', '5'],
 'CMP_cell': ['7'],
 'CM_CD4_T_cell': ['18', '20', '22', '24'],
 'CM_CD8_T_cell': ['25'],
 'Core_CD14_monocyte': ['19', '20', '28', '30'],
 'Core_CD16_monocyte': ['14', '15', '17', '19'],
 'Core_memory_B_cell': ['15', '19'],
 'Core_naive_B_cell': ['17', '21'],
 'Core_naive_CD4_T_cell': ['16', '17', '18', '23', '21', '22'],
 'Core_naive_CD8_T_cell': ['16'],
 'DN_T_cell': ['21', '18', '20'],
 'Early_memory_B_cell': ['3', '16'

## Store Relabel Cell Type Clusters

Store list of leiden clusters to relabel to L3 cell types

In [162]:
relabel_df = pd.read_csv("../data/relabel_deepclean_sheet_06_11_24.csv")
relabel_df = relabel_df[['AIFI_L3', 'relabel_name','clusters']]
# reformat cell types
relabel_df['AIFI_L3'] = [format_cell_type(cell_type) for cell_type in relabel_df['AIFI_L3']]
# filter by cell types that need to be relabeled and clusters only
relabel_df = relabel_df.dropna(thresh= 2)
relabel_df

,AIFI_L3,relabel_name,clusters
0,ASDC,ASDC_uk1_B,"0,1,7"
1,Activated_memory_B_cell,Activated memory B cell_uk1,1
2,Adaptive_NK_cell,Adaptive NK cell_uk1_T,"7,11,20,21"
14,CM_CD4_T_cell,"CM CD4 T cell_uk1_CD8,CM CD4 T cell_uk2","12,17"
24,Early_memory_B_cell,Early memory B cell_uk1,"12,13"
27,GZMBneg_CD27pos_EM_CD4_T_cell,GZMB- CD27+ EM CD4 T cell_uk1_CD8,15
28,GZMBneg_CD27neg_EM_CD4_T_cell,GZMB- CD27- EM CD4 T cell_uk1_CD8,6
29,GZMKpos_CD27pos_EM_CD8_T_cell,GZMK+ CD27+ EM CD8 T cell_uk1_gdt,"10,15,17"
37,ILC,ILC_uk1,2
43,ISGpos_memory_CD4_T_cell,ISG+ memory CD4 T cell_uk1_CD8,15


In [98]:
### remove cell types with more than one names to be relabeled
multiple_relabel_cond = relabel_df['AIFI_L3'].str.contains('|'.join(['CM_CD4_T_cell', 'KLRF1pos_GZMBpos_CD27neg_EM_CD8_T_cell']))
relabel_df_single = relabel_df[~multiple_relabel_cond]
relabel_df_multi = relabel_df[multiple_relabel_cond]

In [99]:
relabel_df_single

,AIFI_L3,relabel_name,clusters
0,ASDC,ASDC_uk1_B,"0,1,7"
1,Activated_memory_B_cell,Activated memory B cell_uk1,1
2,Adaptive_NK_cell,Adaptive NK cell_uk1_T,"7,11,20,21"
24,Early_memory_B_cell,Early memory B cell_uk1,"12,13"
27,GZMBneg_CD27pos_EM_CD4_T_cell,GZMB- CD27+ EM CD4 T cell_uk1_CD8,15
28,GZMBneg_CD27neg_EM_CD4_T_cell,GZMB- CD27- EM CD4 T cell_uk1_CD8,6
29,GZMKpos_CD27pos_EM_CD8_T_cell,GZMK+ CD27+ EM CD8 T cell_uk1_gdt,"10,15,17"
37,ILC,ILC_uk1,2
43,ISGpos_memory_CD4_T_cell,ISG+ memory CD4 T cell_uk1_CD8,15
53,KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell,KLRF1-_GZMB+_CD27-_EM_CD8_uk1,20


In [100]:
relabel_df_multi

,AIFI_L3,relabel_name,clusters
14,CM_CD4_T_cell,"CM CD4 T cell_uk1_CD8,CM CD4 T cell_uk2","12,17"
51,KLRF1pos_GZMBpos_CD27neg_EM_CD8_T_cell,"[KLRF1+_GZMB+_CD27-_EM_CD8_uk1],[KLRF1+_GZMB+_...","[30],[23],[1,4,15,16]"


In [83]:
# Create the dictionary
relabel_cluster_dict = {}

for index, row in relabel_df_single.iterrows():
    key = row['AIFI_L3']
    # parse clusters that have multiple relabel name within same cell types
    cl = row['clusters'].split(',')
    parsed_cl =  [cluster.strip('[]').split(',') for cluster in cl]
    value = (row['relabel_name'].split(','), cl)
    relabel_cluster_dict[key] = value



In [84]:
print(relabel_cluster_dict)

{'ASDC': (['ASDC_uk1_B'], ['0', '1', '7']), 'Activated_memory_B_cell': (['Activated memory B cell_uk1'], ['1']), 'Adaptive_NK_cell': (['Adaptive NK cell_uk1_T'], ['7', '11', '20', '21']), 'Early_memory_B_cell': (['Early memory B cell_uk1'], ['12', '13']), 'GZMBneg_CD27pos_EM_CD4_T_cell': (['GZMB- CD27+ EM CD4 T cell_uk1_CD8'], ['15']), 'GZMBneg_CD27neg_EM_CD4_T_cell': (['GZMB- CD27- EM CD4 T cell_uk1_CD8'], ['6']), 'GZMKpos_CD27pos_EM_CD8_T_cell': (['GZMK+ CD27+ EM CD8 T cell_uk1_gdt'], ['10', '15', '17']), 'ILC': (['ILC_uk1'], ['2']), 'ISGpos_memory_CD4_T_cell': (['ISG+ memory CD4 T cell_uk1_CD8'], ['15']), 'KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell': (['KLRF1-_GZMB+_CD27-_EM_CD8_uk1'], ['20']), 'KLRF1neg_GZMBpos_CD27neg_memory_CD4_T_cell': (['KLRF1-_GZMB+_CD27-_mem_CD4_uk1'], ['5', '6']), 'Proliferating_T_cell': (['Prolif_T_uk1'], ['16']), 'Type_2_polarized_memory_B_cell': (['T2MBC_uk1'], ['19'])}


In [150]:
### manually add cell types with multiple renaming labels to dictionary
multi_relabel_dict = {
    'KLRF1pos_GZMBpos_CD27neg_EM_CD8_T_cell': (
        ['KLRF1+_GZMB+_CD27-_EM_CD8_uk1', 'KLRF1+_GZMB+_CD27-_EM_CD8_uk2', 'KLRF1+GZMB+_CD27-_EM_CD8_uk3'],
        [['30'], ['23'], ['1','4','15','16']]),
    'CM_CD4_T_cell': (
        ['CM CD4 T cell_uk1_CD8', 'CM CD4 T cell_uk2'], 
        [['12'],['17']])}
multi_relabel_dict

{'KLRF1pos_GZMBpos_CD27neg_EM_CD8_T_cell': (['KLRF1+_GZMB+_CD27-_EM_CD8_uk1',
   'KLRF1+_GZMB+_CD27-_EM_CD8_uk2',
   'KLRF1+GZMB+_CD27-_EM_CD8_uk3'],
  [['30'], ['23'], ['1', '4', '15', '16']]),
 'CM_CD4_T_cell': (['CM CD4 T cell_uk1_CD8', 'CM CD4 T cell_uk2'],
  [['12'], ['17']])}

## Process Each Cell Types

In [122]:
### create unit test: cell type with no renaming, cell type with 1 rename and cell type with multiple rename
#subfile_dict = dict((k, file_dict[k]) for k in ('CD4_MAIT', 'ASDC', 'CM_CD4_T_cell'))
subfile_dict = dict((k, file_dict[k]) for k in ('CD4_MAIT', 'ASDC', 'CM_CD4_T_cell'))

subfile_dict

{'CD4_MAIT': '../../05-clustering/scripts/output/preRA_cluster_harmonize_CD4_MAIT_2024-05-25.h5ad',
 'ASDC': '../../05-clustering/scripts/output/preRA_cluster_harmonize_ASDC_2024-05-25.h5ad',
 'CM_CD4_T_cell': '../../05-clustering/scripts/output/preRA_cluster_harmonize_CM_CD4_T_cell_2024-05-26.h5ad'}

In [92]:
out_files = []
out_files

[]

In [94]:
# Initialize out_files as a list at the beginning
out_files = []

# Initialize a list to store each meta DataFrame
meta_list = []

# Iterate over the dictionary containing file names
for label, file in file_dict.items():
    print("Key:", label)
    print("Value:", file)

    # Read the h5ad file
    adata = sc.read_h5ad(file)
    print(adata)

    # Subset doublet dictionary by current cell types
    doublet_clust = doublet_cluster_dict.get(label, [])
    print('Doublet clusters: ' + str(doublet_clust))

    #### Label doublets in metadata
    if len(doublet_clust) == 0:
        print("The list is empty")
        adata.obs['doublets_manual'] = 'no'
    else:
        adata.obs['doublets_manual'] = ['yes' if leiden in doublet_clust else 'no' for leiden in adata.obs['leiden_2']]
    
    # Check the distribution of doublets
    print(pd.crosstab(adata.obs['doublets_manual'], adata.obs['leiden_2']))

    ### Export doublet metadata
    meta = adata.obs[['barcodes', 'batch_id', 'cell_name', 'sample.sampleKitGuid','subject.subjectGuid', 'sample.visitName', 'pbmc_sample_id', 'AIFI_L3', 'leiden_2','doublets_manual']]
    meta.head()

    # Append the current meta DataFrame to the list
    meta_list.append(meta)
    
    # Export the metadata to CSV
    meta.to_csv('./deepcleaned_output/up1_doublet_meta_{c}_{d}.csv'.format(
        c=label,
        d=date.today()
    ))

    # Subset anndata by singlets only
    adata_subset = adata[adata.obs['doublets_manual'] == 'no']
    print("Subsetted cells to: " + str(adata.shape))

    # Export the cleaned anndata
    print('Saving processed data')
    out_file = './deepcleaned_output/up1_deepcleaned_{c}_{d}.h5ad'.format(
        c=label,
        d=date.today()
    )
    adata_subset.write_h5ad(out_file)

    # Append to the output file list
    out_files.append(out_file)


Key: Core_CD14_monocyte
Value: /home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_Core_CD14_monocyte_2024-09-30.h5ad
AnnData object with n_obs × n_vars = 224790 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'specimens.specimenGuid', 'specimens.specimenType', 'file.id', 'file.batchID', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_5

/opt/conda/lib/python3.10/site-packages/anndata/_core/anndata.py:1292: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


Key: KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell
Value: /home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell_2024-09-30.h5ad
AnnData object with n_obs × n_vars = 90994 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'specimens.specimenGuid', 'specimens.specimenType', 'file.id', 'file.batchID', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_

/opt/conda/lib/python3.10/site-packages/anndata/_core/anndata.py:1292: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


Key: CD14pos_cDC2
Value: /home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_CD14pos_cDC2_2024-09-30.h5ad
AnnData object with n_obs × n_vars = 9288 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'specimens.specimenGuid', 'specimens.specimenType', 'file.id', 'file.batchID', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'to

...

/opt/conda/lib/python3.10/site-packages/anndata/_core/anndata.py:1292: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


Key: DN_T_cell
Value: /home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_DN_T_cell_2024-09-30.h5ad
AnnData object with n_obs × n_vars = 8281 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'specimens.specimenGuid', 'specimens.specimenType', 'file.id', 'file.batchID', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_co

/opt/conda/lib/python3.10/site-packages/anndata/_core/anndata.py:1292: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


Key: Naive_Vd1_gdT
Value: /home/jupyter/certpro_up1/unharmonized_output/up1_cluster_unharmonize_Naive_Vd1_gdT_2024-09-30.h5ad
AnnData object with n_obs × n_vars = 40534 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'specimens.specimenGuid', 'specimens.specimenType', 'file.id', 'file.batchID', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 

/opt/conda/lib/python3.10/site-packages/anndata/_core/anndata.py:1292: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


In [111]:
out_files
out_files

['./deepcleaned_output_2/up1_deepcleaned_Core_CD14_monocyte_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_CD14pos_cDC2_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_GZMKpos_CD27pos_EM_CD8_T_cell_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_ISGpos_MAIT_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_CD56bright_NK_cell_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_CM_CD8_T_cell_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_SOX4pos_naive_CD4_T_cell_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_CMP_cell_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_IL1Bpos_CD14_monocyte_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_Activated_memory_B_cell_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_ASDC_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_CD4_MAIT_2024-10-16.

In [96]:
path = '/home/jupyter/certpro_up1/deepcleaned_output/'

In [97]:
out_files_csv = get_filepaths_with_glob(path, "up1_doublet_meta_*.csv") 
out_files_csv

['/home/jupyter/certpro_up1/deepcleaned_output/up1_doublet_meta_BaEoMaP_cell_2024-10-16.csv',
 '/home/jupyter/certpro_up1/deepcleaned_output/up1_doublet_meta_GZMKpos_CD56dim_NK_cell_2024-10-16.csv',
 '/home/jupyter/certpro_up1/deepcleaned_output/up1_doublet_meta_GZMKneg_CD56dim_NK_cell_2024-10-16.csv',
 '/home/jupyter/certpro_up1/deepcleaned_output/up1_doublet_meta_GZMKpos_memory_CD4_Treg_2024-10-16.csv',
 '/home/jupyter/certpro_up1/deepcleaned_output/up1_doublet_meta_KLRF1pos_effector_Vd1_gdT_2024-10-16.csv',
 '/home/jupyter/certpro_up1/deepcleaned_output/up1_doublet_meta_ISGpos_naive_CD4_T_cell_2024-10-16.csv',
 '/home/jupyter/certpro_up1/deepcleaned_output/up1_doublet_meta_ISGpos_CD56dim_NK_cell_2024-10-16.csv',
 '/home/jupyter/certpro_up1/deepcleaned_output/up1_doublet_meta_Core_memory_B_cell_2024-10-16.csv',
 '/home/jupyter/certpro_up1/deepcleaned_output/up1_doublet_meta_CM_CD8_T_cell_2024-10-16.csv',
 '/home/jupyter/certpro_up1/deepcleaned_output/up1_doublet_meta_ASDC_2024-10-16.

## Export Doublet Metadata

In [98]:
doublet_meta_comb = pd.concat(meta_list)

In [99]:
### check metadata
len(doublet_meta_comb['AIFI_L3'].unique())

70

In [101]:
### export
doublet_meta_comb.to_csv("./deepcleaned_output/up1_scRNA_218_samples_doublet_metadata_all_types.csv")

## Upload Cell Type data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [102]:
ss = hisepy.get_study_spaces()
study_space_uuid = ss[0]['id']
title = 'IDE for 06 PBMC L3 Deepclean {d}'.format(d = date.today())
title

'IDE for 06 PBMC L3 Deepclean 2024-10-16'

In [103]:
search_id = element_id()
search_id

'seaborgium-thulium-cerium'

In [104]:
in_files = list(h5ad_uuids.values())
in_files

['7467d353-d8bf-467e-911c-4f759b83caf7',
 'eebab05a-bd3c-4921-8ae3-7345fac966c9',
 '410fc93a-f497-4ea4-81e2-fb8cbdc79a28',
 '6b18c554-64b3-43ae-9514-6ecd18c30f4d',
 '6f5b21ba-d98b-4003-bcf0-30e26a2cc2df',
 '153e45fc-e772-497c-9c4b-e986b7885f01',
 '9acd576f-2271-4ddb-b93e-20f0f63f87af',
 'fcafe12f-3a15-4807-aaea-e93f08445d02',
 '66122892-cddf-4c40-88b2-eaa53b7a6f4c',
 'c9356501-0f48-4590-83ea-34da6dc72b56',
 '0942c4af-20b7-4e2c-a822-a2e8f5044fff',
 'e8dfca7a-e64e-469b-bcc8-6d32a291ffed',
 'caad876a-5c80-4d5a-b245-c6c112b40912',
 'defe9422-db72-4210-b933-6f8919b001c5',
 'b2eec8c5-5bf9-47df-9428-ce26878f017d',
 'e19e9467-2945-4ddc-ac95-1403cc0aee4b',
 'c78d13d1-9580-4b4f-b212-11890ce22544',
 '60c73f54-4210-463f-9ced-920eb73bf924',
 'b8d55475-5c20-4bdb-8980-edaae687d77d',
 '27701eca-1fc6-4d14-9ed6-099e4d1b2668',
 'c5f11c26-23c5-4288-9200-176d57a52ddd',
 'f54f89d0-340d-47a9-bef2-ebcb854c6d72',
 '7ae41ec4-d970-424c-bba3-f5c9debfb3df',
 '48602a6c-9cb1-491d-a2d6-1075bf989e86',
 '91b76287-3228-

In [105]:
len(in_files)

70

In [110]:
out_files

['./deepcleaned_output_2/up1_deepcleaned_Core_CD14_monocyte_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_CD14pos_cDC2_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_GZMKpos_CD27pos_EM_CD8_T_cell_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_ISGpos_MAIT_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_CD56bright_NK_cell_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_CM_CD8_T_cell_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_SOX4pos_naive_CD4_T_cell_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_CMP_cell_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_IL1Bpos_CD14_monocyte_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_Activated_memory_B_cell_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_ASDC_2024-10-16.h5ad',
 './deepcleaned_output_2/up1_deepcleaned_CD4_MAIT_2024-10-16.

In [112]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

you are trying to upload file_ids... ['./deepcleaned_output_2/up1_deepcleaned_Core_CD14_monocyte_2024-10-16.h5ad', './deepcleaned_output_2/up1_deepcleaned_KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell_2024-10-16.h5ad', './deepcleaned_output_2/up1_deepcleaned_CD14pos_cDC2_2024-10-16.h5ad', './deepcleaned_output_2/up1_deepcleaned_GZMKpos_CD27pos_EM_CD8_T_cell_2024-10-16.h5ad', './deepcleaned_output_2/up1_deepcleaned_ISGpos_MAIT_2024-10-16.h5ad', './deepcleaned_output_2/up1_deepcleaned_CD56bright_NK_cell_2024-10-16.h5ad', './deepcleaned_output_2/up1_deepcleaned_CM_CD8_T_cell_2024-10-16.h5ad', './deepcleaned_output_2/up1_deepcleaned_SOX4pos_naive_CD4_T_cell_2024-10-16.h5ad', './deepcleaned_output_2/up1_deepcleaned_CMP_cell_2024-10-16.h5ad', './deepcleaned_output_2/up1_deepcleaned_IL1Bpos_CD14_monocyte_2024-10-16.h5ad', './deepcleaned_output_2/up1_deepcleaned_Activated_memory_B_cell_2024-10-16.h5ad', './deepcleaned_output_2/up1_deepcleaned_ASDC_2024-10-16.h5ad', './deepcleaned_output_2/up1_deepcle

(y/n) y


{'trace_id': '656083d3-2214-4ae5-92cb-f7aacc40e6e9',
 'files': ['./deepcleaned_output_2/up1_deepcleaned_Core_CD14_monocyte_2024-10-16.h5ad',
  './deepcleaned_output_2/up1_deepcleaned_KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell_2024-10-16.h5ad',
  './deepcleaned_output_2/up1_deepcleaned_CD14pos_cDC2_2024-10-16.h5ad',
  './deepcleaned_output_2/up1_deepcleaned_GZMKpos_CD27pos_EM_CD8_T_cell_2024-10-16.h5ad',
  './deepcleaned_output_2/up1_deepcleaned_ISGpos_MAIT_2024-10-16.h5ad',
  './deepcleaned_output_2/up1_deepcleaned_CD56bright_NK_cell_2024-10-16.h5ad',
  './deepcleaned_output_2/up1_deepcleaned_CM_CD8_T_cell_2024-10-16.h5ad',
  './deepcleaned_output_2/up1_deepcleaned_SOX4pos_naive_CD4_T_cell_2024-10-16.h5ad',
  './deepcleaned_output_2/up1_deepcleaned_CMP_cell_2024-10-16.h5ad',
  './deepcleaned_output_2/up1_deepcleaned_IL1Bpos_CD14_monocyte_2024-10-16.h5ad',
  './deepcleaned_output_2/up1_deepcleaned_Activated_memory_B_cell_2024-10-16.h5ad',
  './deepcleaned_output_2/up1_deepcleaned_ASDC_2024-

In [172]:
import session_info
session_info.show()